# SeeMPS Benchmark

Beware that this notebook takes a few hours to run due to the long-time integration of the diffusion PDE.

Running this notebook will generate a CSV file with the times and accuracies achieved. To plot the results, you must run `03_plot_benchmarks.ipynb` afterwards.

To compare the 'cython' and 'python' cores, one must set the variable `SEEMPS_BACKEND` in the cell below, run the notebook, and restart the kernel before changing the backend. SeeMPS loads the core a single time on the first import. 

In [1]:
# Configuration
# Restart the kernel for backend changes to take effect
SEEMPS_BACKEND = "cython"  # 'cython' or 'python'
BENCH_THREADS = 1
N_REPS = 101
N_VALUES = list(range(4, 15))
TOLERANCE = 1e-6

In [2]:
import os

# Set number of threads
os.environ.setdefault("SEEMPS_BACKEND", SEEMPS_BACKEND)
os.environ.setdefault("OMP_NUM_THREADS", str(BENCH_THREADS))
os.environ.setdefault("MKL_NUM_THREADS", str(BENCH_THREADS))
os.environ.setdefault("OPENBLAS_NUM_THREADS", str(BENCH_THREADS))

import csv
import time
import numpy as np
import scipy.sparse as sp
import scipy.sparse.linalg as spla

# SeeMPS imports
from seemps.analysis.mesh import QuantizedInterval
from seemps.cython import active_backend
from seemps.evolution import tdvp, runge_kutta
from seemps.hamiltonians import ConstantNNHamiltonian
from seemps.optimization import dmrg
from seemps.register.transforms import mpo_weighted_shifts
from seemps.state import DEFAULT_STRATEGY, MPS, CanonicalMPS, product_state, random_mps

LIBRARY = f"seemps-{active_backend()}"
OUTPUT_CSV = f"results_{LIBRARY}.csv"
print(f"Library: {LIBRARY}")

Library: seemps-cython


In [ ]:
# MPS truncation strategy shared by all benchmarks
def strategy(nsites, normalize):
    return DEFAULT_STRATEGY.replace(
        normalize=normalize,
        tolerance=TOLERANCE**2 / (nsites - 1),
        simplification_tolerance=TOLERANCE,
    )


# Transverse Field Ising model
def tfim_hamiltonian(N, g=1.0):
    sz = np.diag([1.0, -1.0]) / 2
    sx = np.array([[0, 1], [1, 0]], dtype=float) / 2
    H = ConstantNNHamiltonian(N, dimension=2)
    for i in range(N - 1):
        H.add_interaction_term(i, -np.kron(sz, sz))
    for i in range(N):
        H.add_local_term(i, -g * sx)
    return H.to_mpo()


# Reference ground-state energy via sparse diagonalization
def reference_ground_energy(H):
    Hmat = sp.csr_matrix(H.to_matrix())
    vals = spla.eigsh(Hmat, k=1, which="SA", return_eigenvectors=False)
    return float(vals[0])


# Laplacian MPO via periodic finite-differences
def laplacian_mpo(Nq, dx):
    return mpo_weighted_shifts(
        Nq, np.array([1, -2, 1]) / dx**2, [-1, 0, 1], periodic=True
    )

In [4]:
# Ground state energy via DMRG
def benchmark_dmrg(N, nreps):
    H = tfim_hamiltonian(N)
    E0 = reference_ground_energy(H)
    stgy = strategy(N, normalize=True)

    # Initial condition
    np.random.seed(0)
    guess = CanonicalMPS(
        random_mps([2] * N, D=4, complex=False), center=0, normalize=True
    )

    # Run benchmark
    times, errs = [], []
    for _ in range(nreps):
        t0 = time.perf_counter()
        result = dmrg(H, guess=guess, maxiter=20, tol=TOLERANCE, strategy=stgy)
        times.append(time.perf_counter() - t0)
        errs.append(abs(result.energy - E0) / max(abs(E0), 1.0))

    return "hamiltonian", "dmrg", N, result.state.max_bond_dimension(), times, errs


# Time evolution via 2-site TDVP
def benchmark_tdvp(N, nreps):
    H = tfim_hamiltonian(N)
    psi0 = product_state(np.array([1.0, 1.0]) / np.sqrt(2.0), length=N)
    T, steps = 0.05, 50
    stgy = strategy(N, normalize=False)

    # Exact solution
    Hmat = sp.csr_matrix(H.to_matrix())
    exact = spla.expm_multiply(-1j * T * Hmat, psi0.to_vector())
    exact_norm = np.linalg.norm(exact)

    # Run benchmark
    times, errs = [], []
    for _ in range(nreps):
        t0 = time.perf_counter()
        psi_T = tdvp(-1j * H, (0.0, T), psi0, steps=steps, strategy=stgy)
        times.append(time.perf_counter() - t0)
        errs.append(np.linalg.norm(psi_T.to_vector() - exact) / exact_norm)

    return "hamiltonian", "tdvp", N, psi_T.max_bond_dimension(), times, errs


# Diffusion PDE via RK4
def benchmark_pde(N, nreps):
    interval = QuantizedInterval(0.0, 2.0, N, endpoint_right=False)
    x, dx, nu = interval.to_vector(), interval.step, 0.01
    stgy = strategy(N, normalize=False)

    # Initial condition
    x0 = 1.0
    sigma = 0.1
    psi_vec = np.exp(-((x - x0) ** 2) / (2 * sigma**2))
    psi0 = MPS.from_vector(psi_vec, [2] * N, strategy=stgy, normalize=False)

    # Setup
    D2 = laplacian_mpo(N, dx)
    steps = 1_000
    dt = 0.1 * (0.5 * dx**2 / nu)  # dt from stability
    T = steps * dt

    # Analytic solution
    sigma_t = np.sqrt(sigma**2 + 2 * nu * T)
    psi_exact = np.exp(-((x - x0) ** 2) / (2 * sigma_t**2)) * sigma / sigma_t

    # Run benchmark
    times, errs = [], []
    for _ in range(nreps):
        t0 = time.perf_counter()
        psi = runge_kutta(nu * D2, (0.0, T), psi0, steps=steps, strategy=stgy)
        times.append(time.perf_counter() - t0)
        v = psi.to_vector()
        errs.append(np.linalg.norm(v - psi_exact) / np.linalg.norm(psi_exact))

    return "pde", "diffusion_rk4", N, psi.max_bond_dimension(), times, errs

In [5]:
BENCHMARKS = [benchmark_dmrg, benchmark_tdvp, benchmark_pde]

# Warmup
for fn in BENCHMARKS:
    fn(N_VALUES[0], 1)

# Real benchmark
records = []
for N in N_VALUES:
    for fn in BENCHMARKS:
        problem, subproblem, n, D, times, errs = fn(N, N_REPS)
        print(
            f"{subproblem:14s} N={n} D={D}  t={np.median(times):.4f}s  err={np.median(errs):.3e}"
        )
        records.append(
            dict(
                problem=problem,
                subproblem=subproblem,
                library=LIBRARY,
                N=n,
                D=D,
                time_samples=";".join(f"{v}" for v in times),
                accuracy_samples=";".join(f"{v}" for v in errs),
            )
        )

# Save results
with open(OUTPUT_CSV, "w", newline="") as f:
    w = csv.DictWriter(f, fieldnames=list(records[0]))
    w.writeheader()
    w.writerows(records)
print(f"Wrote {len(records)} records to {OUTPUT_CSV}")

dmrg           N=4 D=4  t=0.0017s  err=6.362e-16
tdvp           N=4 D=2  t=0.0494s  err=4.340e-10
diffusion_rk4  N=4 D=3  t=0.8061s  err=7.434e-01
dmrg           N=5 D=4  t=0.0026s  err=3.382e-16
tdvp           N=5 D=2  t=0.0743s  err=6.137e-10
diffusion_rk4  N=5 D=4  t=1.1758s  err=1.619e-01
dmrg           N=6 D=6  t=0.0034s  err=1.406e-15
tdvp           N=6 D=2  t=0.0968s  err=7.516e-10
diffusion_rk4  N=6 D=6  t=1.6694s  err=4.323e-03
dmrg           N=7 D=6  t=0.0042s  err=3.972e-15
tdvp           N=7 D=2  t=0.1185s  err=8.679e-10
diffusion_rk4  N=7 D=6  t=2.2501s  err=5.380e-04
dmrg           N=8 D=6  t=0.0051s  err=2.293e-14
tdvp           N=8 D=2  t=0.1409s  err=9.704e-10
diffusion_rk4  N=8 D=6  t=2.7898s  err=1.548e-04
dmrg           N=9 D=6  t=0.0059s  err=7.325e-14
tdvp           N=9 D=2  t=0.1640s  err=1.063e-09
diffusion_rk4  N=9 D=7  t=3.5758s  err=1.876e-05
dmrg           N=10 D=7  t=0.0069s  err=9.512e-14
tdvp           N=10 D=2  t=0.1853s  err=1.148e-09
diffusion_rk4  N=1